#  ⭐ `fat_reacoes`


In [94]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, mapping_column, normalize_date_column, normalize_rows, duration_normalize, expandir_gravidade_wide

In [95]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [96]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

### Head

In [97]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17


### GRAVE

In [98]:
bronze.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

### DESFECHO

In [99]:
bronze.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

### GRAVIDADE

In [100]:
bronze.GRAVIDADE.value_counts()

GRAVIDADE
Outro efeito clinicamente significativo                                                                                                                                                                              144833
Hospitalização/Prolongamento de hospitalização                                                                                                                                                                        57343
Hospitalização/Prolongamento de hospitalização, Outro efeito clinicamente significativo                                                                                                                               15621
Resultou em óbito                                                                                                                                                                                                     11510
Ameaça à vida                                                                                                 

In [101]:
 
gravidades_unicas = (
    bronze['GRAVIDADE']
    .dropna()
    .astype(str)
    .str.split(r'\s*,\s*')   # separa pelos ","
    .explode()               # uma linha por item
    .str.strip()             # tira espaços extras
    .drop_duplicates()       # remove duplicados
    .sort_values()           # opcional: ordena
)

print(gravidades_unicas.to_list())


['Ameaça à vida', 'Anomalia congênita ou malformação ao nascer', 'Hospitalização/Prolongamento de hospitalização', 'Incapacidade persistente ou significativa', 'Outro efeito clinicamente significativo', 'Resultou em óbito']


In [102]:
for g in gravidades_unicas:
    print(g)

Ameaça à vida
Anomalia congênita ou malformação ao nascer
Hospitalização/Prolongamento de hospitalização
Incapacidade persistente ou significativa
Outro efeito clinicamente significativo
Resultou em óbito


### DURACAO

In [103]:
bronze.DURACAO.value_counts().head(10)

DURACAO
1 dia        27059
2 dia        10017
3 dia         6650
1 hora        6314
30 minuto     5865
4 dia         4278
5 dia         3707
0 dia         3349
2 hora        3112
20 minuto     2698
Name: count, dtype: int64

# 🥈 Silver

normalized data, medium quality


In [104]:
silver = bronze.copy()

In [105]:
dim_soc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc/dim_soc.parquet")
dim_soc.head()

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


### pk

In [106]:
cols = ['IDENTIFICACAO_NOTIFICACAO', 'PK_SOC', 'PK_LLT','DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO']
hist_fat_reacoes = silver.merge(dim_soc, on=['REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT', 'HLT', 'HLGT','SOC'], how='left')
hist_fat_reacoes = hist_fat_reacoes[cols]
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,9,9107,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,16,3903,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,12,11411,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,8,1994,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


#### date

In [107]:

col_dates = ["DATA_INICIO_HORA", "DATA_FINAL_HORA"]

for col in col_dates:
    if col in hist_fat_reacoes.columns:
        hist_fat_reacoes[col] = normalize_date_column(hist_fat_reacoes[col])

hist_fat_reacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815877 entries, 0 to 815876
Data columns (total 10 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO  815877 non-null  object        
 1   PK_SOC                     770393 non-null  Int64         
 2   PK_LLT                     815877 non-null  int64         
 3   DATA_INICIO_HORA           347696 non-null  datetime64[ns]
 4   DATA_FINAL_HORA            189475 non-null  datetime64[ns]
 5   DURACAO                    127726 non-null  object        
 6   GRAVE                      642796 non-null  object        
 7   GRAVIDADE                  267804 non-null  object        
 8   DESFECHO                   680205 non-null  object        
 9   ATUALIZADO                 815877 non-null  object        
dtypes: Int64(1), datetime64[ns](2), int64(1), object(6)
memory usage: 63.0+ MB


In [108]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


### GRAVE

In [109]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

In [110]:
 
grave_map = {
    "Sim": 1,
    "Não": 2, 
}
hist_fat_reacoes['GRAVE'] = hist_fat_reacoes['GRAVE'].map(grave_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes[cols]
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,2,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,1,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,1,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,1,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,1,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


In [111]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
2    374682
1    268114
0    173081
Name: count, dtype: Int64

###  DESFECHO

In [112]:
hist_fat_reacoes.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

In [113]:
desfecho_map = {
    "Desconhecido": 0, 
    "Não Recuperado/Não Resolvido/Em andamento": 1, 
    "Em recuperação/Resolvendo": 2, 
    "Recuperado/Resolvido": 3,
    "Fatal/Óbito": 4, 
    "Recuperado/Resolvido com sequelas": 5, 
}
hist_fat_reacoes['DESFECHO'] = hist_fat_reacoes['DESFECHO'].map(desfecho_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes[cols]
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,2,None,3,2025-11-17
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,1,Outro efeito clinicamente significativo,3,2025-11-17
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,1,Outro efeito clinicamente significativo,3,2025-11-17
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,1,Outro efeito clinicamente significativo,3,2025-11-17
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,1,Hospitalização/Prolongamento de hospitalização,3,2025-11-17


In [114]:
hist_fat_reacoes.DESFECHO.value_counts()

DESFECHO
0    362166
3    278670
1     85241
2     66908
4     18581
5      4311
Name: count, dtype: Int64

### DURACAO

In [115]:
hist_fat_reacoes.DURACAO.value_counts().head(10)

DURACAO
1 dia        27059
2 dia        10017
3 dia         6650
1 hora        6314
30 minuto     5865
4 dia         4278
5 dia         3707
0 dia         3349
2 hora        3112
20 minuto     2698
Name: count, dtype: int64

In [116]:
hist_fat_reacoes = duration_normalize(hist_fat_reacoes, col='DURACAO', prefix='DURACAO')


In [117]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,DURACAO_VALOR,DURACAO_TIPO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,2,None,3,2025-11-17,3.0,dia
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,1,Outro efeito clinicamente significativo,3,2025-11-17,NaN,none
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,1,Outro efeito clinicamente significativo,3,2025-11-17,2.0,dia
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,1,Outro efeito clinicamente significativo,3,2025-11-17,5.0,dia
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,1,Hospitalização/Prolongamento de hospitalização,3,2025-11-17,NaN,none


In [118]:
hist_fat_reacoes['DURACAO_TIPO'] = hist_fat_reacoes['DURACAO_TIPO'].fillna("Desconhecido")
hist_fat_reacoes.DURACAO_TIPO.value_counts()

DURACAO_TIPO
none            688151
dia              76020
minuto           24232
hora             17868
mês               5644
semana            2259
ano               1313
segundo            353
Desconhecido        24
década              13
Name: count, dtype: int64

In [119]:
hist_fat_reacoes.DURACAO_VALOR.value_counts().head(15)

DURACAO_VALOR
1.0     37128
2.0     15479
3.0      9917
30.0     6422
5.0      6190
4.0      6146
10.0     4270
6.0      3670
0.0      3481
20.0     3372
7.0      3269
15.0     2735
8.0      2652
40.0     1979
12.0     1612
Name: count, dtype: int64

### GRAVIDADE

In [121]:
hist_fat_reacoes = expandir_gravidade_wide(hist_fat_reacoes, col='GRAVIDADE', prefix='GRAVIDADE_')
hist_fat_reacoes = hist_fat_reacoes.drop('GRAVIDADE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,DESFECHO,ATUALIZADO,DURACAO_VALOR,DURACAO_TIPO,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,2,3,2025-11-17,3.0,dia,0,0,0,0,0,0
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,1,3,2025-11-17,NaN,none,0,0,0,0,1,0
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,1,3,2025-11-17,2.0,dia,0,0,0,0,1,0
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,1,3,2025-11-17,5.0,dia,0,0,0,0,1,0
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,1,3,2025-11-17,NaN,none,0,0,0,1,0,0


### SAVE

In [123]:
hist_fat_reacoes.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'PK_SOC', 'PK_LLT', 'DATA_INICIO_HORA',
       'DATA_FINAL_HORA', 'DURACAO', 'GRAVE', 'DESFECHO', 'ATUALIZADO',
       'DURACAO_VALOR', 'DURACAO_TIPO', 'GRAVIDADE_RESULTADO_OBITO',
       'GRAVIDADE_AMEACA_VIDA', 'GRAVIDADE_INCAPACIDADE',
       'GRAVIDADE_HOSPITALIZACAO', 'GRAVIDADE_OUTRO_EFEITO',
       'GRAVIDADE_ANOMALIA_CONGENITA'],
      dtype='object')

In [124]:
hist_fat_reacoes[['IDENTIFICACAO_NOTIFICACAO', 'PK_SOC', 'PK_LLT', 
                  'DATA_INICIO_HORA','DATA_FINAL_HORA',
                  'GRAVE','DESFECHO', 
                  'DURACAO_VALOR', 'DURACAO_TIPO',
                  'GRAVIDADE_RESULTADO_OBITO', 'GRAVIDADE_AMEACA_VIDA', 'GRAVIDADE_INCAPACIDADE', 'GRAVIDADE_HOSPITALIZACAO', 
                  'GRAVIDADE_ANOMALIA_CONGENITA','GRAVIDADE_OUTRO_EFEITO'
       ]].to_parquet(Path(project_root) / "data/02_silver/hist_fat_reacoes/hist_fat_reacoes.parquet", index=False)

# 🥇 Gold


In [125]:
fat_reacoes = hist_fat_reacoes.copy()
fat_reacoes.to_parquet(Path(project_root) / "data/03_gold/fat_reacoes/fat_reacoes.parquet", index=False)